In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
catalog_name = "ecommerce"

In [0]:
df_bromze = spark.table(f"{catalog_name}.bronze.btr_brands")
df_bromze.show(10)

In [0]:
df_silver = df_bromze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r"[^A-Za-z0-9]", ""))
df_silver.show(10)

In [0]:

anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(anomalies, subset="category_code")
df_silver.show(40)
df_silver.select("category_code").distinct().show()


In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupby("category_code").count().filter("count > 1")
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display (df_silver)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

In [0]:

df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")
row_count, column_count = df_bronze.count(), len(df_bronze.columns)
print (f"Row count: {row_count}")
print (f"Column count: {column_count}")

In [0]:
df_silver = df_bronze.withColumn(
"weight_grams",
F.regexp_replace(F.col ("weight_grams"), "g", "").cast (IntegerType())
)
df_silver.select("weight_grams") .show(5, truncate=False)

In [0]:
df_silver = df_silver.withColumn(
"length_cm",
F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(3)

In [0]:
df_silver = df_silver.withColumn(
"category_code",
F.upper(F.col("category_code"))
).withColumn(
"brand_code",
F.upper(F.col("brand_code"))
)
df_silver.select("category_code", "brand_code").show(2)

In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
"material",
F.when(F.col("material") == "Coton", "Cotton")
.when(F.col("material") == "Alumium", "Aluminum")
.when(F.col("material") == "Ruber", "Rubber")
.otherwise(F.col("material"))
)
df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
"rating_count",
F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
.otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
df_silver.select(
"weight_grams",
"length_cm",
"category_code",
"brand_code",
"material",
"rating_count"
).show(10, truncate=False)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")
# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)
# Print the results
print (f"Row count: {row_count}")
print (f"Column count: {column_count}")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.dropna(subset=["customer_id"])
# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
df_silver = df_silver.fillna("Not Available", subset=["phone"])
# sanity check (If any nulls still exist)
df_silver.filter(F.col("phone").isNull()).show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")
# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)
# Print the results
print (f"Row count: {row_count}")
print (f"Column count: {column_count}")
df_bronze.show(3)

In [0]:
from pyspark.sql.functions import to_date

# Convert the string column to a date type
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

In [0]:
duplicates = df_silver.groupBy('date').count().filter("count > 1")
# Show the duplicate rows
print ("Total duplicated Rows: "
, duplicates. count ())
display(duplicates)

In [0]:
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit(" "), F.col("year"))))
df_silver = df_silver.withColumn("week_of year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit(""), F.col("year"))))
df_silver.show(3)

In [0]:
df_silver = df_silver.drop("week_of_year").withColumnRenamed("week_of year", "week_of_year")

In [0]:
df_silver.write.format("delta") \
.mode("overwrite") \
.option("mergeSchema", "true") \
.saveAsTable(f"{catalog_name}.silver.slv_calendar")